In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, to_timestamp, upper, trim
from pyspark.sql.window import Window
from delta.tables import DeltaTable


CDC_PATH = (
    "abfss://source@stshopsensedevhj.dfs.core.windows.net/"
    "bronze/cdc/orders/"
)

SILVER_PATH = (
    "abfss://source@stshopsensedevhj.dfs.core.windows.net/"
    "silver/orders/"
)


# Read Bronze files
cdc_raw = spark.read.parquet(CDC_PATH)

total_rows = cdc_raw.count()

if total_rows == 0:
    dbutils.notebook.exit("NO_CHANGES")

print(f"[BRONZE] Rows read: {total_rows}")


# Keep latest row per OrderID using LastModifiedDate
latest_window = (
    Window
    .partitionBy("OrderID")
    .orderBy(F.desc("LastModifiedDate"))
)

cdc_latest = (
    cdc_raw
    .withColumn("_rn", F.row_number().over(latest_window))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

latest_count = cdc_latest.count()

print(f"[LATEST] Latest OrderID rows: {latest_count}")


def transform_orders(df):
    return (
        df
        .withColumn("OrderDate", to_timestamp("OrderDate"))
        .withColumn("ShippedDate", to_timestamp("ShippedDate"))
        .withColumn("DeliveredDate", to_timestamp("DeliveredDate"))
        .withColumn(
            "LastModifiedDate",
            to_timestamp("LastModifiedDate")
        )
        .withColumn(
            "TotalAmount",
            col("TotalAmount").cast("decimal(10,2)")
        )
        .withColumn(
            "DiscountAmount",
            col("DiscountAmount").cast("decimal(10,2)")
        )
        .withColumn(
            "ShippingCharges",
            col("ShippingCharges").cast("decimal(10,2)")
        )
        .withColumn(
            "OrderStatus",
            upper(trim(col("OrderStatus")))
        )
        .withColumn(
            "PaymentMethod",
            upper(trim(col("PaymentMethod")))
        )
        .withColumn(
            "IsPrimeOrder",
            upper(trim(col("IsPrimeOrder").cast("string"))) == "TRUE"
        )
        .withColumn("OrderYear", F.year("OrderDate"))
        .withColumn("OrderMonth", F.month("OrderDate"))
        .withColumn("OrderDayOfWeek", F.dayofweek("OrderDate"))
        .withColumn(
            "IsWeekendOrder",
            col("OrderDayOfWeek").isin([1, 7])
        )
        .withColumn(
            "IsDelivered",
            col("OrderStatus") == "DELIVERED"
        )
        .withColumn(
            "IsCancelled",
            col("OrderStatus") == "CANCELLED"
        )
        .withColumn(
            "IsReturned",
            col("OrderStatus") == "RETURNED"
        )
        .withColumn(
            "NetAmount",
            col("TotalAmount")
            - col("DiscountAmount")
            + col("ShippingCharges")
        )
        .withColumn(
            "DaysToDeliver",
            F.when(
                col("DeliveredDate").isNotNull(),
                F.datediff(
                    col("DeliveredDate"),
                    col("OrderDate")
                )
            )
        )
        .withColumn(
            "_silver_load_ts",
            F.current_timestamp()
        )
        .withColumn(
            "_source",
            F.lit("adf_cdc_parquet")
        )
        .withColumn(
            "_is_deleted",
            F.lit(False)
        )
    )


if not DeltaTable.isDeltaTable(spark, SILVER_PATH):
    raise Exception(
        "Silver Orders table does not exist. "
        "Run initial load first."
    )


silver = DeltaTable.forPath(spark, SILVER_PATH)

transformed_orders = transform_orders(cdc_latest)


(
    silver.alias("s")
    .merge(
        transformed_orders.alias("c"),
        "s.OrderID = c.OrderID"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print(f"[MERGE] {latest_count} rows merged into Silver.")


final_df = spark.read.format("delta").load(SILVER_PATH)

print(f"[DONE] Silver Orders rows: {final_df.count()}")

[BRONZE] Rows read: 27
[LATEST] Latest OrderID rows: 21
[MERGE] 21 rows merged into Silver.
[DONE] Silver Orders rows: 3015


In [0]:
display(
    final_df
    .filter(
        col("OrderID").isin(
            "ORD_CDC_106",
            "ORD_CDC_107",
            "ORD_CDC_108",
            "ORD_CDC_109",
            "ORD_CDC_110",
            "ORD_CDC_111",
            "ORD_CDC_112",
            "ORD_CDC_113",
            "ORD_CDC_114",
            "ORD_CDC_115"
        )
    )
    .select(
        "OrderID",
        "OrderStatus",
        "LastModifiedDate",
        "_is_deleted"
    )
    .orderBy("OrderID")
)

OrderID,OrderStatus,LastModifiedDate,_is_deleted
ORD_CDC_106,SHIPPED,2026-07-15T05:19:13.993333Z,false
ORD_CDC_107,SHIPPED,2026-07-15T05:19:13.993333Z,false
ORD_CDC_108,SHIPPED,2026-07-15T05:19:13.993333Z,false
ORD_CDC_109,SHIPPED,2026-07-15T05:19:13.993333Z,false
ORD_CDC_110,SHIPPED,2026-07-15T05:19:13.993333Z,false
ORD_CDC_111,PROCESSING,2026-07-15T05:33:25.463333Z,false
ORD_CDC_112,PROCESSING,2026-07-15T05:33:25.463333Z,false
ORD_CDC_113,PROCESSING,2026-07-15T05:33:25.463333Z,false
ORD_CDC_114,PROCESSING,2026-07-15T05:33:25.463333Z,false
ORD_CDC_115,PROCESSING,2026-07-15T05:33:25.463333Z,false
